In [6]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils.DataLoader import DataLoader

from transformers import BertTokenizer, BertModel
from tqdm import tqdm


In [12]:
# dataloader = DataLoader(r'.\ctg-studies.json')
# df = dataloader.load_data()
# # df.head()
# df.to_csv(r'.\ctg-studies.csv', index=False)

In [2]:
df = pd.read_csv(r'.\ctg-studies.csv')
nctid_eligibility = df[['NCT ID', 'Eligibility Criteria']]
nctid_eligibility.head()

,NCT ID,Eligibility Criteria
0,NCT00019136,DISEASE CHARACTERISTICS:\n\n* Histologically p...
1,NCT00085930,Inclusion Criteria:\n\nHigh risk neuroblastoma...
2,NCT00881920,INCLUSION CRITERIA:\n\nBLOOD PROCUREMENT:\n\n*...
3,NCT00902044,INCLUSION CRITERIA:\n\nProcurement Eligibility...
4,NCT00924326,* INCLUSION CRITERIA:\n\n 1. Patient must hav...


In [8]:
# # Load model directly
# from transformers import AutoTokenizer, AutoModelForMaskedLM

# tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")
# model = AutoModelForMaskedLM.from_pretrained("medicalai/ClinicalBERT")

In [9]:
tokenizer = BertTokenizer.from_pretrained("medicalai/ClinicalBERT")
model = BertModel.from_pretrained("medicalai/ClinicalBERT")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DistilBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
You are using a model of type distilbert to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
Some weights of BertModel were not initialized from the model checkpoint at medicalai/ClinicalBERT and are newly initialized: ['encoder.layer.4.attention.self.query.bias', 'encoder.layer.5.output.LayerNorm.weight', 'encoder.layer.10.attention.output.dense.bias', 'encoder.layer.6.intermediate.dense.bias', 'encoder.layer.10.attention.self.key.bias', 'encoder.layer.7.attention.self.value.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.3.intermediate.dense.weight', 'encoder.layer.2.output.LayerNorm.bias', 'encoder.layer.9.a

In [11]:
# Tokenize and get embeddings for each eligibility criteria
embeddings = []
max_len = 512
for text in tqdm(nctid_eligibility['Eligibility Criteria']):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_len)
    outputs = model(**inputs)
    embeddings.append(outputs.pooler_output.detach().numpy())

# Flatten the embeddings
feature_matrix = [emb.flatten() for emb in embeddings]

 45%|████▌     | 576/1268 [28:46<34:33,  3.00s/it]  


ValueError: text input must of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).